[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/03-OCRfacturas.ipynb)


# 03 — Comparando métodos de OCR: de lo clásico a un modelo de Hugging Face

En **`02-PDF_reporte`** vimos que `pypdf` solo sirve si el PDF ya trae texto seleccionable. En cuanto el "documento" es en realidad una foto, `extract_text()` regresa (casi) nada — la información sigue ahí, pero como **píxeles**, no como caracteres.

Aquí comparamos **cuatro formas distintas** de resolver eso sobre el mismo problema real: **extraer los datos de una factura** (número, fecha, proveedor, total) a partir de su imagen.

| Ronda | Método | Qué es |
|---|---|---|
| 1 | **Tesseract** | El motor de OCR "clásico", sin deep learning. |
| 2 | **EasyOCR** | OCR con redes neuronales, entrenado para texto "del mundo real". |
| 3 | **PaddleOCR** | OCR moderno, más robusto con layouts distintos. |
| 4 | **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** | Modelo de Hugging Face especializado en documentos (300M parámetros): entiende que una tabla es una tabla y te la regresa ya estructurada. |

Corremos las cuatro rondas sobre **las mismas 3 facturas** y comparamos contra el dato real — lo sabemos porque son parte de un dataset de facturas sintéticas con respuesta conocida. Al final, con Falcon-OCR, probamos algo más: **¿importa cómo capturaste el documento?** (PDF renderizado limpio vs. una foto del PDF).


## Carpeta `public/`

Igual que en `02-PDF_reporte`, este notebook espera una carpeta **`public/`** junto al notebook con los archivos que se usan. Todo el código de aquí en adelante asume que ya existen ahí — nada de detectar si estás en Colab o en local, es la misma carpeta en los dos casos.

**En Colab:** crea la carpeta `public` (panel de archivos → clic derecho → *Nueva carpeta*) y arrastra ahí:

- `batch1-0081.jpg`, `batch1-0039.jpg`, `batch1-0001.jpg` — 3 facturas del dataset de Kaggle [**High-Quality Invoice Images for OCR**](https://www.kaggle.com/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr) (son de un tercero, no vienen en el repo).
- `ey_100_casos_rentables_ia_2026.pdf` — el mismo PDF de `02-PDF_reporte`.
- `fakeine.jpeg` — para el bonus opcional del final.

**En local:** si clonaste el repo, `modulo_4/public/` ya trae las 3 facturas, el PDF y el fake INE.


In [ ]:
from pathlib import Path

CARPETA = Path("public")
RUTAS_FACTURAS = {
    "batch1-0081.jpg": CARPETA / "batch1-0081.jpg",
    "batch1-0039.jpg": CARPETA / "batch1-0039.jpg",
    "batch1-0001.jpg": CARPETA / "batch1-0001.jpg",
}


## El dato real

Como es un dataset con respuesta conocida, guardamos aquí los 4 campos correctos de cada factura. Así, en vez de "eyeballear" si cada método funcionó, lo comparamos automáticamente al final.


In [ ]:
DATO_REAL = {
    "batch1-0081.jpg": {
        "invoice_number": "15288019",
        "invoice_date": "09/07/2014",
        "seller_name": "Fernandez Ltd",
        "total": "1.71",
    },
    "batch1-0039.jpg": {
        "invoice_number": "35593328",
        "invoice_date": "06/08/2011",
        "seller_name": "Cruz-Carter",
        "total": "3004.96",
    },
    "batch1-0001.jpg": {
        "invoice_number": "51109338",
        "invoice_date": "04/13/2013",
        "seller_name": "Andrews, Kirby and Valdez",
        "total": "6204.19",
    },
}


## Ronda 1 — Tesseract

**Tesseract** no usa deep learning: detecta formas de caracteres contra patrones. Es el motor de OCR "de toda la vida".

**Ventajas:** gratis, instantáneo, no necesita GPU ni internet (una vez instalado).

**Desventajas:** solo texto plano, sin noción de estructura (tablas, columnas); le cuesta con fotos giradas o de mala calidad; tú tienes que armar los campos con regex.


In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
%pip install -q pytesseract


In [ ]:
import time

import pytesseract
from PIL import Image


def ocr_tesseract(ruta):
    return pytesseract.image_to_string(Image.open(ruta))


resultados_tesseract = {}
for nombre, ruta in RUTAS_FACTURAS.items():
    t0 = time.time()
    texto = ocr_tesseract(ruta)
    resultados_tesseract[nombre] = {"texto": texto, "segundos": time.time() - t0}

for nombre, r in resultados_tesseract.items():
    print(f"--- {nombre} ({r['segundos']:.2f}s) ---")
    print(r["texto"][:400])
    print()


## De texto plano a campos: una regex mínima

Las 3 facturas usan **la misma plantilla** (es un dataset sintético), así que dos patrones bastan para sacar el número, la fecha y el total de cualquier texto plano.

Ojo: el total vive dentro de una **tabla** (`Total  $ neto  $ IVA  $ bruto`). El OCR plano no siempre respeta el orden de las columnas al leer una tabla, así que es normal que el total falle más seguido que el número o la fecha — sobre todo en la factura de 7 renglones. Ese es justo el problema que resuelve Falcon-OCR en la Ronda 4.


In [ ]:
import re


def extraer_campos(texto):
    numero = re.search(r"Invoice no:?\s*(\d+)", texto)
    fecha = re.search(r"Date of issue:?\s*([\d/]+)", texto)
    vendedor = re.search(r"Seller:?\s*\n?\s*([A-Za-z,.&\- ]+)", texto)
    total = re.findall(r"\$\s*(\d[\d.,\s]*\d|\d)", texto)

    return {
        "invoice_number": numero.group(1) if numero else None,
        "invoice_date": fecha.group(1) if fecha else None,
        "seller_name": vendedor.group(1).strip() if vendedor else None,
        # el último "$" de la tabla suele ser el Gross worth == total
        "total": total[-1] if total else None,
    }


## Función para comparar contra el dato real

La reutilizamos para las 4 rondas.


In [ ]:
import pandas as pd


def normalizar(valor):
    if valor is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())


def campo_correcto(extraido, real):
    return normalizar(real) != "" and normalizar(real) in normalizar(extraido)


def evaluar_metodo(nombre_metodo, resultados, extractor):
    """resultados: {nombre_factura: {'texto': ..., 'segundos': ...}}"""
    filas = []
    for nombre_factura, r in resultados.items():
        real = DATO_REAL[nombre_factura]
        campos = extractor(r)
        fila = {"metodo": nombre_metodo, "factura": nombre_factura, "segundos": round(r["segundos"], 2)}
        for campo in ("invoice_number", "invoice_date", "seller_name", "total"):
            fila[campo] = "✅" if campo_correcto(campos.get(campo), real[campo]) else "❌"
        filas.append(fila)
    return pd.DataFrame(filas)


In [ ]:
tabla_tesseract = evaluar_metodo(
    "Tesseract", resultados_tesseract, lambda r: extraer_campos(r["texto"])
)
tabla_tesseract


## Ronda 2 — EasyOCR

OCR con redes neuronales, entrenado para texto "del mundo real" (facturas, tickets, letreros). Mismo procedimiento, mismas 3 facturas, mismo extractor de campos — solo cambia el motor.

**Ventajas:** mejor que Tesseract con fotos reales (ángulos, fondos, fuentes variadas).

**Desventajas:** más lento, descarga un modelo de ~500 MB la primera vez, y sigue sin entender que hay una tabla — el mismo problema de columnas que Tesseract.


In [ ]:
%pip install -q easyocr


In [ ]:
import easyocr

lector_easyocr = easyocr.Reader(["en"], gpu=False)


def ocr_easyocr(ruta):
    lineas = lector_easyocr.readtext(str(ruta), detail=0)
    return "\n".join(lineas)


resultados_easyocr = {}
for nombre, ruta in RUTAS_FACTURAS.items():
    t0 = time.time()
    texto = ocr_easyocr(ruta)
    resultados_easyocr[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_easyocr = evaluar_metodo(
    "EasyOCR", resultados_easyocr, lambda r: extraer_campos(r["texto"])
)
tabla_easyocr


## Ronda 3 — PaddleOCR

Otro motor con deep learning, con mejor manejo de rotación y de layouts variados que EasyOCR.

**Ventajas:** más robusto ante fotos giradas o con layouts distintos entre sí.

**Desventajas:** instalación más pesada; con el modo simple que usamos aquí, tampoco resuelve tablas complejas (para eso existe **PP-StructureV3**, que ya reconstruye tablas completas — vale la pena explorarlo si quieren ir más lejos).


In [ ]:
%pip install -q paddlepaddle paddleocr


In [ ]:
from paddleocr import PaddleOCR

# Si tu version de paddleocr ya no acepta use_angle_cls/lang asi, revisa
# la documentacion actual del paquete instalado (la API ha cambiado entre versiones).
lector_paddle = PaddleOCR(use_angle_cls=True, lang="en")


def ocr_paddle(ruta):
    resultado = lector_paddle.ocr(str(ruta), cls=True)
    lineas = [linea[1][0] for bloque in resultado for linea in bloque]
    return "\n".join(lineas)


resultados_paddle = {}
for nombre, ruta in RUTAS_FACTURAS.items():
    t0 = time.time()
    texto = ocr_paddle(ruta)
    resultados_paddle[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_paddle = evaluar_metodo(
    "PaddleOCR", resultados_paddle, lambda r: extraer_campos(r["texto"])
)
tabla_paddle


## Ronda 4 — Falcon-OCR, un modelo de Hugging Face

Hasta aquí sacamos **texto plano** y lo peinamos con regex. **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** (de TII) es distinto: es un modelo chico (300M parámetros, ~3× más chico que otros modelos de su categoría) **especializado en documentos**, no un chatbot genérico. Sabe hacer tres cosas según se lo pidas: texto plano, fórmulas en LaTeX, o **tablas en HTML ya estructuradas**.

**Necesita GPU.** Antes de seguir: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

Vamos a hacer exactamente lo que harían en su trabajo: entrar a la página del modelo en Hugging Face, copiar el código de uso rápido ("Quickstart") de la ficha del modelo, y pegarlo aquí. Es este:


In [ ]:
%pip install -q -U transformers accelerate


In [ ]:
# Codigo de la ficha del modelo (https://huggingface.co/tiiuae/Falcon-OCR), copiado y pegado casi tal cual
# -- solo cambiamos la ruta de la imagen por una de nuestras facturas.
import torch
from PIL import Image
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/Falcon-OCR",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

image = Image.open("public/batch1-0081.jpg")
texts = model.generate(image)  # categoria por default: texto plano
print(texts[0])


Eso es todo lo que pedía la ficha del modelo. Con el modelo ya cargado en memoria (`model`), lo corremos sobre nuestras 3 facturas y reutilizamos el mismo `extraer_campos` de las rondas anteriores.


In [ ]:
resultados_falcon = {}
for nombre, ruta in RUTAS_FACTURAS.items():
    t0 = time.time()
    texto = model.generate(Image.open(ruta))[0]
    resultados_falcon[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_falcon = evaluar_metodo(
    "Falcon-OCR", resultados_falcon, lambda r: extraer_campos(r["texto"])
)
tabla_falcon


### El truco: pedirle la tabla directo

Falcon-OCR también sabe leer `category="table"` y regresar la tabla de renglones **ya en HTML**, en vez de texto plano donde las columnas se revuelven. Probémoslo con la factura de 7 renglones — la que más le costó trabajo al regex del total en las rondas anteriores.


In [ ]:
from IPython.display import HTML, display

tabla_html = model.generate(Image.open(RUTAS_FACTURAS["batch1-0001.jpg"]), category="table")[0]
print(tabla_html)
display(HTML(tabla_html))


Esa es la ventaja real de un modelo especializado en documentos: no tuvimos que escribir ni una regex para reconstruir la tabla — se la pedimos directo.

**Ventajas:** entiende que una tabla es una tabla (HTML estructurado, sin regex); modelo chico (300M) y especializado, no un LLM genérico de propósito general.

**Desventajas:** necesita GPU — no es viable en CPU; no es conversacional, solo tiene 3 modos fijos (texto/fórmula/tabla), no le puedes "preguntar" cosas libres como a un chatbot.


### ¿Importa cómo capturaste el documento? PDF renderizado vs. foto del PDF

Todo lo anterior fue con **fotos ya digitales** (el dataset de facturas). Pero en la vida real, muchas veces el punto de partida es un **PDF** (como el de `02-PDF_reporte`) y alguien lo **imprime y le toma una foto con el celular** en vez de mandar el archivo digital.

Corremos **Falcon-OCR** sobre **la misma página**, capturada de dos formas:

1. **PDF renderizado limpio** — convertimos la página del PDF a imagen directamente.
2. **"Foto del PDF"** — simulamos una foto de esa misma página ya impresa: ángulo, sombra/reflejo, menos resolución, compresión JPEG.


In [ ]:
%pip install -q pypdfium2

import pypdfium2 as pdfium

pdf_ey = pdfium.PdfDocument("public/ey_100_casos_rentables_ia_2026.pdf")
pagina_limpia = pdf_ey[0].render(scale=2).to_pil().convert("RGB")
pagina_limpia


In [ ]:
import numpy as np


def simular_foto(imagen, angulo=6, escala=0.5, calidad_jpeg=40):
    """Aproxima una foto de celular a un documento impreso: rotación, menos resolución,
    sombra/gradiente de luz y compresión JPEG agresiva."""
    from io import BytesIO

    img = imagen.rotate(angulo, expand=True, fillcolor=(255, 255, 255))
    nuevo_ancho = int(img.width * escala)
    nuevo_alto = int(img.height * escala)
    img = img.resize((nuevo_ancho, nuevo_alto))

    # Gradiente de sombra/reflejo de izquierda a derecha
    gradiente = np.tile(np.linspace(0.65, 1.0, nuevo_ancho), (nuevo_alto, 1))
    arreglo = np.array(img).astype(float)
    for canal in range(3):
        arreglo[:, :, canal] *= gradiente
    img = Image.fromarray(np.clip(arreglo, 0, 255).astype("uint8"))

    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=calidad_jpeg)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


pagina_foto = simular_foto(pagina_limpia)
pagina_foto


In [ ]:
texto_limpio_pdf = model.generate(pagina_limpia)[0]
texto_foto_pdf = model.generate(pagina_foto)[0]

print("--- PDF renderizado limpio ---")
print(texto_limpio_pdf[:400])
print()
print("--- Foto simulada del PDF ---")
print(texto_foto_pdf[:400])


Si la segunda transcripción tiene más errores, palabras cortadas o directamente le faltan pedazos, esa es la lección: **el modelo importa, pero la calidad de la captura del documento importa tanto o más.** Es la misma razón por la que, en producción, casi siempre conviene pedir el PDF/archivo original en vez de aceptar una foto.


## Comparando todo

Juntamos las tablas de las 4 rondas.


In [ ]:
comparacion = pd.concat(
    [tabla_tesseract, tabla_easyocr, tabla_paddle, tabla_falcon], ignore_index=True
)
comparacion


In [ ]:
resumen = comparacion.groupby("metodo").agg(
    segundos_promedio=("segundos", "mean"),
    aciertos_numero=("invoice_number", lambda s: (s == "✅").sum()),
    aciertos_fecha=("invoice_date", lambda s: (s == "✅").sum()),
    aciertos_vendedor=("seller_name", lambda s: (s == "✅").sum()),
    aciertos_total=("total", lambda s: (s == "✅").sum()),
)
resumen


## Cierre

Cada ronda tuvo su tabla de ventajas/desventajas arriba — el resumen: entre más entiende el modelo de **estructura** (tabla, layout), menos regex necesitas escribir tú. Y no menos importante: **la captura del documento importa tanto como el método** — un PDF limpio y una foto borrosa del mismo PDF no le dan la misma información a ningún método.

En producción, muchas empresas ya resuelven esto con **servicios administrados** (Google Document AI, AWS Textract, Azure Document Intelligence) que ya vienen entrenados para facturas — la ventaja de hacerlo "a mano" aquí es entender **qué está pasando por dentro** antes de delegarlo a una caja negra.


## Bonus opcional — otro tipo de documento: una identificación

*(Sáltate esta sección si vas corto de tiempo — es solo para mostrar que la misma técnica generaliza a otro tipo de documento, no solo facturas.)*

Reutilizamos el modelo ya cargado (Falcon-OCR) sobre una credencial **de ejemplo, generada para fines didácticos** (`public/fakeine.jpeg`) — no es un documento real.


In [ ]:
texto_ine = model.generate(Image.open("public/fakeine.jpeg"))[0]
print(texto_ine)


Aquí no hay una plantilla fija como en las facturas, así que ya no alcanza con el `extraer_campos` de antes — tocaría escribir patrones nuevos para nombre, CURP, fecha de nacimiento, etc. Con Tesseract o EasyOCR sería el mismo trabajo extra. Falcon-OCR al menos ya te dio el texto limpio y ordenado como punto de partida.
